#SETUP

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["PERPLEXITY_API_KEY"] = os.getenv("PERPLEXITY_API_KEY")
os.environ["OPENAI_API_KEY"] = "dummy"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"
print("Environment variables loaded successfully.")


Environment variables loaded successfully.


#LLM RUNS

In [4]:
import os
from dotenv import load_dotenv
from langsmith import traceable
from langchain_perplexity import ChatPerplexity
load_dotenv()
PKEY = os.getenv("PERPLEXITY_API_KEY")
model = ChatPerplexity(api_key=PKEY, model="sonar-pro")

@traceable(run_type="llm", metadata={"provider": "perplexity"})
def chat_model(messages: list):
    """
    messages: list of dicts containing role + content.
    Returns: assistant text.
    """
    resp = model.invoke(messages)
    content = getattr(resp, "content", None) or str(resp)
    return content
inputs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello! Can you summarise LangChain tracing?"}
]
answer = chat_model(inputs)
print("Assistant reply:\n", answer)


Assistant reply:
 LangChain tracing is a feature that records and visualizes the step-by-step operations performed by every component (such as language models, retrievers, or tools) during the execution of a chain or agent. This provides developers with detailed observability into how inputs are processed and outputs are generated, making debugging, optimization, and monitoring significantly easier[1][3][9].

**Key aspects of LangChain tracing:**

- **Activity Recording:** Each component involved in a workflow (like an LLM, retriever, or custom tool) automatically logs its actions to a tracer[1].
- **Data Capturing:** Traces include inputs, outputs, execution time, and any errors or exceptions for each step[1][3].
- **Trace Structure:** A "trace" is a collection of related "runs" for a single user operation. For example, a trace for a prompt might contain runs for the prompt template, LLM call, and output parser[3].
- **Visualization:** Traces can be viewed in dashboards (e.g., LangSmi

#HANDLING STREAMING LLM RUNS

In [6]:
from langsmith import traceable
file_url = "file:///mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png"
def _reduce_chunks(chunks: list):
    """Concatenate chunk texts into one LLM-like final response object."""
    all_text = "".join(chunk["choices"][0]["message"]["content"] for chunk in chunks)
    return {"choices": [{"message": {"content": all_text, "role": "assistant"}}]}
@traceable(
    run_type="llm",
    metadata={"ls_provider": "perplexity-demo", "ls_model_name": "streaming-demo"},
    reduce_fn=_reduce_chunks,
)
def my_streaming_chat_model(messages: list):
    """
    Simulated streaming generator.
    Yields a short analysis in two chunks about an OHLC market note referenced by file_url.
    """
    user_text = messages[1]["content"] if len(messages) > 1 else "No user prompt provided."
    yield {
        "choices": [
            {"message": {"content": "Market summary: The recent candles show higher highs and higher lows, ", "role": "assistant"}}
        ]
    }
    yield {
        "choices": [
            {"message": {"content": "indicating a short-term bullish bias (see reference: " + file_url + ")", "role": "assistant"}}
        ]
    }
inputs = [
    {"role": "system", "content": "You are a concise market analyst."},
    {"role": "user", "content": f"Please read the OHLC chart note and give a short market bias (bullish/bearish) and one-sentence rationale. Reference: {file_url}"}
]
chunks = list(my_streaming_chat_model(inputs))

print("Chunks (streamed):")
for i, c in enumerate(chunks, 1):
    text = c["choices"][0]["message"]["content"]
    print(f"  chunk {i} -> {repr(text)}")
reduced = _reduce_chunks(chunks)
final_text = reduced["choices"][0]["message"]["content"]
print("\nReduced (final) text:\n", final_text)


Chunks (streamed):
  chunk 1 -> 'Market summary: The recent candles show higher highs and higher lows, '
  chunk 2 -> 'indicating a short-term bullish bias (see reference: file:///mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png)'

Reduced (final) text:
 Market summary: The recent candles show higher highs and higher lows, indicating a short-term bullish bias (see reference: file:///mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png)


#RETRIEVER RUNS

In [8]:
from langsmith import traceable
LOCAL_PATH = "/mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png"  # plain text only

def _convert(results):
    return [
        {
            "page_content": text,
            "type": "Document",
            "metadata": {"source": "demo", "local_path": LOCAL_PATH}
        }
        for text in results
    ]
@traceable(run_type="retriever")
def retrieve_docs(query: str):
    results = [
        f"Result 1 about: {query}",
        f"Result 2 mentioning: {query}",
        f"Result 3 extracted from analysis of: {query}"
    ]
    return _convert(results)
docs = retrieve_docs("market structure analysis")
docs


[{'page_content': 'Result 1 about: market structure analysis',
  'type': 'Document',
  'metadata': {'source': 'demo',
   'local_path': '/mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png'}},
 {'page_content': 'Result 2 mentioning: market structure analysis',
  'type': 'Document',
  'metadata': {'source': 'demo',
   'local_path': '/mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png'}},
 {'page_content': 'Result 3 extracted from analysis of: market structure analysis',
  'type': 'Document',
  'metadata': {'source': 'demo',
   'local_path': '/mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png'}}]

#TOOL CALLING

In [9]:
import os, json
from dotenv import load_dotenv
from langsmith import traceable
from langchain_perplexity import ChatPerplexity
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "dummy-key"
PKEY = os.getenv("PERPLEXITY_API_KEY")
model = ChatPerplexity(api_key=PKEY, model="sonar-pro")
file_url = "file:///mnt/data/570b7b58-f96b-4394-84e6-73c7046db8e4.png"
@traceable(run_type="tool")
def get_distance(origin: str, destination: str):
    pairs = {("Delhi","Mumbai"): {"distance_km":1400,"approx_time_hrs":24},("Mumbai","Delhi"): {"distance_km":1400,"approx_time_hrs":24}}
    key = (origin.strip().title(), destination.strip().title())
    r = pairs.get(key, {"distance_km":None,"approx_time_hrs":None})
    return {"origin": origin, "destination": destination, **r}
@traceable(run_type="llm")
def call_llm(messages, tools=None):
    return model.invoke(messages)
@traceable(run_type="chain")
def ask_distance(inputs, tools):
    resp = call_llm(inputs, tools)
    tool_call = None
    try:
        tool_call = resp.choices[0].message.tool_calls[0]
        args_json = tool_call.function.arguments
    except Exception:
        content = getattr(resp, "content", "") or str(resp)
        start, end = content.find("{"), content.rfind("}")
        args_json = content[start:end+1] if start!=-1 and end>start else None
    if args_json:
        try:
            tool_args = json.loads(args_json)
            origin = tool_args.get("origin")
            destination = tool_args.get("destination")
        except Exception:
            return resp
    else:
        return resp
    tool_result = get_distance(origin, destination)
    tool_response_message = {"role":"tool","content": json.dumps({"tool":"get_distance","result":tool_result,"file_url":file_url}), "tool_call_id": getattr(tool_call, "id", None)}
    inputs.append(resp.choices[0].message if hasattr(resp, "choices") else {"role":"assistant","content":getattr(resp,"content",str(resp))})
    inputs.append(tool_response_message)
    final = call_llm(inputs, None)
    return final

tools = [
    {"type":"function","function":{"name":"get_distance","description":"Return approximate distance/time between two cities","parameters":{"type":"object","properties":{"origin":{"type":"string"},"destination":{"type":"string"},"file_url":{"type":"string"}},"required":["origin","destination"]}}}
]
inputs = [
    {"role":"system","content":"You are a helpful assistant that can call a distance tool."},
    {"role":"user","content":f"How far is it to travel from Delhi to Mumbai? Use a tool if needed. Reference file: {file_url}"}
]
final = ask_distance(inputs, tools)
try:
    print(final.choices[0].message.content)
except Exception:
    print(getattr(final, "content", final))


The travel distance from **Delhi to Mumbai** depends on the mode of transportation:

- **By air (direct flight):** The aerial distance is approximately **1,134–1,148 km**[1][2][3].
- **By road:** The distance ranges from about **1,331 km** to **1,408 km**[3][5].
- **By train (main rail line):** The railway line covers about **1,386 km**[7].

Typical travel times are:
- **Flight:** Around 2 to 3.25 hours (nonstop)[1][4].
- **Driving:** About 16 to 17 hours, depending on traffic and route[2][5].
- **Train:** Approximately 15.5 to 18 hours[5][4].

These distances are consistent with the routes shown in your referenced map file. The exact distance may vary slightly based on the specific starting and ending points within each city and the chosen route.
